In [16]:
# Imports

import os
import json
import faiss
import numpy as np
import pandas as pd

from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer

from langchain_core.documents import Document
from langchain_core.chat_history import InMemoryChatMessageHistory

from langchain_groq import ChatGroq

In [ ]:
# Loading Environment Variables 

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Groq API Key not found")

print("Groq API Loaded Successfully")

In [18]:
# Company data 

companies_df = pd.DataFrame({

    "company_name":[
        "Acme Corp",
        "Globex",
        "Initech",
        "Umbrella Health",
        "Wayne Enterprises"
    ],

    "industry":[
        "Manufacturing",
        "Retail",
        "Software",
        "Healthcare",
        "Industrial"
    ],

    "strategic_focus":[
        "AI Logistics",
        "Customer Analytics",
        "Cloud Migration",
        "Patient Analytics",
        "Predictive Maintenance"
    ],

    "recent_news":[
        "Expanding into Asia",
        "Launching loyalty platform",
        "Cloud modernization initiative",
        "Patient analytics platform",
        "Smart factory investments"
    ]
})

In [19]:
# meeting notes 

meeting_notes_df = pd.DataFrame({

    "client":[
        "Acme Corp",
        "Acme Corp",
        "Globex"
    ],

    "date":[
        "2025-02-10",
        "2025-04-12",
        "2025-05-01"
    ],

    "notes":[
        "Discussed AI logistics implementation.",
        "Requested pricing proposal.",
        "Reviewed analytics roadmap."
    ]
})

In [20]:
# action items 

action_items_df = pd.DataFrame({

    "client":[
        "Acme Corp",
        "Acme Corp",
        "Globex"
    ],

    "action":[
        "Send Proposal",
        "Schedule Workshop",
        "Share Case Study"
    ],

    "status":[
        "Open",
        "Open",
        "Closed"
    ]
})

In [21]:
#contacts 

contacts_df = pd.DataFrame({

    "client":[
        "Acme Corp",
        "Acme Corp"
    ],

    "name":[
        "Sarah Johnson",
        "Michael Chen"
    ],

    "role":[
        "Procurement Director",
        "CTO"
    ]
})

In [22]:
# short term memory 

chat_history = InMemoryChatMessageHistory()

In [23]:
# long term memory 

MEMORY_FILE = "memory.json"

#Create Memory File

def save_memory(data):

    with open(MEMORY_FILE, "w") as f:
        json.dump(data, f, indent=4)

#Load Memory

def load_memory():

    with open(MEMORY_FILE, "r") as f:
        return json.load(f)

#Initialize

if not os.path.exists(MEMORY_FILE):

    default_memory = {

        "preferred_style":"concise",

        "important_clients":[],

        "meeting_history":[],

        "user_preferences":[]
    }

    save_memory(default_memory)

In [24]:
# creating knowledge based documents 

documents = []

for _, row in companies_df.iterrows():

    text = f"""
    Company Name: {row['company_name']}
    Industry: {row['industry']}
    Strategic Focus: {row['strategic_focus']}
    Recent News: {row['recent_news']}
    """

    documents.append(
        Document(page_content=text)
    )

#CONVERTING TO TEXTS
texts = [
    doc.page_content
    for doc in documents
]

print(texts)

['\n    Company Name: Acme Corp\n    Industry: Manufacturing\n    Strategic Focus: AI Logistics\n    Recent News: Expanding into Asia\n    ', '\n    Company Name: Globex\n    Industry: Retail\n    Strategic Focus: Customer Analytics\n    Recent News: Launching loyalty platform\n    ', '\n    Company Name: Initech\n    Industry: Software\n    Strategic Focus: Cloud Migration\n    Recent News: Cloud modernization initiative\n    ', '\n    Company Name: Umbrella Health\n    Industry: Healthcare\n    Strategic Focus: Patient Analytics\n    Recent News: Patient analytics platform\n    ', '\n    Company Name: Wayne Enterprises\n    Industry: Industrial\n    Strategic Focus: Predictive Maintenance\n    Recent News: Smart factory investments\n    ']


In [ ]:
# generate embeddings 

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    texts
)

print(embeddings.shape)

In [ ]:
# building faiss index 

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(
        embeddings,
        dtype=np.float32
    )
)
print("Total Number of Index created:", index.ntotal)

In [27]:
# company search 

def company_search(company_name):

    query_embedding = embedding_model.encode(
        [company_name]
    )

    distances, indices = index.search(
        np.array(
            query_embedding,
            dtype=np.float32
        ),
        3
    )

    results = []

    for idx in indices[0]:
        results.append(texts[idx])

    return "\n".join(results)

In [28]:
# meeting notes 

def meeting_notes(company_name):

    result = meeting_notes_df[
        meeting_notes_df["client"]
        ==
        company_name
    ]

    return result.to_string(index=False)

In [29]:
# action items 

def action_items(company_name):

    result = action_items_df[
        action_items_df["client"]
        ==
        company_name
    ]

    return result.to_string(index=False)

In [30]:
# contacts 

def contacts(company_name):

    result = contacts_df[
        contacts_df["client"]
        ==
        company_name
    ]

    return result.to_string(index=False)

In [31]:
# memory tool 

def memory_tool():

    return json.dumps(
        load_memory(),
        indent=2
    )

In [32]:
# tool schemas 

AVAILABLE_TOOLS = [

    "company_search",

    "meeting_notes",

    "action_items",

    "contacts",

    "memory_tool"
]

In [ ]:
# planner agent 

# initializing groq

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="llama-3.3-70b-versatile",
    temperature=0.1
)

In [33]:
# planner prompt 

PLANNER_PROMPT = """
You are a planning agent.

Available tools:

1. company_search
2. meeting_notes
3. action_items
4. contacts
5. memory_tool

Return JSON only.

Example:

{
    "tools":[
        {
            "tool_name":"company_search",
            "company_name":"Acme Corp"
        }
    ]
}
"""

In [ ]:
# planner function 

def create_plan(user_query):

    prompt = f"""
    {PLANNER_PROMPT}

    User Query:
    {user_query}
    """

    response = llm.invoke(prompt)

    print("Planner Response:")
    print(response.content)

    return response.content

In [34]:
import json
import re

def parse_plan(plan_text):

    try:

        # Remove markdown fences
        plan_text = plan_text.replace(
            "```json",
            ""
        )

        plan_text = plan_text.replace(
            "```",
            ""
        )

        plan_text = plan_text.strip()

        # Extract JSON object
        match = re.search(
            r"\{.*\}",
            plan_text,
            re.DOTALL
        )

        if match:
            plan_text = match.group()

        return json.loads(plan_text)

    except Exception as e:

        print("Plan Parsing Error:")
        print(e)

        print("\nRaw Response:")
        print(plan_text)

        return {
            "tools":[]
        }

In [35]:
# tool executor 

def execute_tool(tool_name, company_name=None):

    if tool_name == "company_search":
        return company_search(company_name)

    elif tool_name == "meeting_notes":
        return meeting_notes(company_name)

    elif tool_name == "action_items":
        return action_items(company_name)

    elif tool_name == "contacts":
        return contacts(company_name)

    elif tool_name == "memory_tool":
        return memory_tool()

    return "Unknown Tool"

In [36]:
# execute plan 

def execute_plan(plan):

    parsed = parse_plan(plan)

    tool_results = {}

    for step in parsed["tools"]:

        tool_name = step["tool_name"]

        company_name = step.get(
            "company_name"
        )

        tool_results[tool_name] = execute_tool(
            tool_name,
            company_name
        )

    return tool_results

In [37]:
# meeting brief generator 

def generate_meeting_brief(
    user_query,
    tool_results
):

    prompt = f"""
    Create an executive meeting brief.

    User Query:
    {user_query}

    Retrieved Context:

    {json.dumps(tool_results, indent=2)}

    Include:

    1. Company Overview
    2. Previous Meetings
    3. Open Action Items
    4. Stakeholders
    5. Risks
    6. Opportunities
    7. Talking Points
    8. Next Steps
    """

    response = llm.invoke(prompt)

    return response.content

In [38]:
# main agent workflow

def meeting_preparation_agent(
    user_query
):

    chat_history.add_user_message(
        user_query
    )

    plan = create_plan(
        user_query
    )

    tool_results = execute_plan(
        plan
    )

    final_response = (
        generate_meeting_brief(
            user_query,
            tool_results
        )
    )

    chat_history.add_ai_message(
        final_response
    )

    return final_response

In [ ]:
# running end to end demo 

user_query = "Prepare me for my meeting with Acme Corp"

response = meeting_preparation_agent(user_query
)

print(response)